# Exploratory data analysis with SQL

Ten executed SQLite queries on the separate 101-row course SQL snapshot. TRIM merges a trailing-space variant of Success. Mission success differs from landing success.

Executed reconstruction created for this report. These are not recovered copies of previously unshared learner notebooks.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json
from IPython.display import display, Image, HTML
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)


In [2]:
import sqlite3
con=sqlite3.connect(':memory:');df=pd.read_csv(DATA/'Spacex.csv');df.to_sql('launches',con,index=False);print('Records:',len(df))

Records: 101


In [3]:
query='SELECT DISTINCT Launch_Site FROM launches ORDER BY Launch_Site'
print(query)
display(pd.read_sql_query(query,con))

SELECT DISTINCT Launch_Site FROM launches ORDER BY Launch_Site


,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


In [4]:
query="SELECT Date,Booster_Version,Launch_Site FROM launches WHERE Launch_Site LIKE 'CCA%' ORDER BY Date LIMIT 5"
print(query)
display(pd.read_sql_query(query,con))

SELECT Date,Booster_Version,Launch_Site FROM launches WHERE Launch_Site LIKE 'CCA%' ORDER BY Date LIMIT 5


,Date,Booster_Version,Launch_Site
0,2010-06-04,F9 v1.0 B0003,CCAFS LC-40
1,2010-12-08,F9 v1.0 B0004,CCAFS LC-40
2,2012-05-22,F9 v1.0 B0005,CCAFS LC-40
3,2012-10-08,F9 v1.0 B0006,CCAFS LC-40
4,2013-03-01,F9 v1.0 B0007,CCAFS LC-40


In [5]:
query="SELECT SUM(PAYLOAD_MASS__KG_) AS kg FROM launches WHERE Customer='NASA (CRS)'"
print(query)
display(pd.read_sql_query(query,con))

SELECT SUM(PAYLOAD_MASS__KG_) AS kg FROM launches WHERE Customer='NASA (CRS)'


,kg
0,45596


In [6]:
query="SELECT AVG(PAYLOAD_MASS__KG_) AS kg FROM launches WHERE Booster_Version LIKE 'F9 v1.1%'"
print(query)
display(pd.read_sql_query(query,con))

SELECT AVG(PAYLOAD_MASS__KG_) AS kg FROM launches WHERE Booster_Version LIKE 'F9 v1.1%'


,kg
0,2534.666667


In [7]:
query="SELECT MIN(Date) AS Date FROM launches WHERE Landing_Outcome='Success (ground pad)'"
print(query)
display(pd.read_sql_query(query,con))

SELECT MIN(Date) AS Date FROM launches WHERE Landing_Outcome='Success (ground pad)'


,Date
0,2015-12-22


In [8]:
query="SELECT DISTINCT Booster_Version FROM launches WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000"
print(query)
display(pd.read_sql_query(query,con))

SELECT DISTINCT Booster_Version FROM launches WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000


,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


In [9]:
query='SELECT TRIM(Mission_Outcome) AS Mission_Outcome, COUNT(*) AS n FROM launches GROUP BY TRIM(Mission_Outcome)'
print(query)
display(pd.read_sql_query(query,con))

SELECT TRIM(Mission_Outcome) AS Mission_Outcome, COUNT(*) AS n FROM launches GROUP BY TRIM(Mission_Outcome)


,Mission_Outcome,n
0,Failure (in flight),1
1,Success,99
2,Success (payload status unclear),1


In [10]:
query='SELECT Booster_Version,PAYLOAD_MASS__KG_ FROM launches WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM launches)'
print(query)
display(pd.read_sql_query(query,con))

SELECT Booster_Version,PAYLOAD_MASS__KG_ FROM launches WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM launches)


,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


In [11]:
query="SELECT Date,Booster_Version,Launch_Site FROM launches WHERE Date LIKE '2015%' AND Landing_Outcome='Failure (drone ship)'"
print(query)
display(pd.read_sql_query(query,con))

SELECT Date,Booster_Version,Launch_Site FROM launches WHERE Date LIKE '2015%' AND Landing_Outcome='Failure (drone ship)'


,Date,Booster_Version,Launch_Site
0,2015-01-10,F9 v1.1 B1012,CCAFS LC-40
1,2015-04-14,F9 v1.1 B1015,CCAFS LC-40


In [12]:
query="SELECT Landing_Outcome,COUNT(*) AS n FROM launches WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY n DESC"
print(query)
display(pd.read_sql_query(query,con))

SELECT Landing_Outcome,COUNT(*) AS n FROM launches WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY n DESC


,Landing_Outcome,n
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
